1) Run cells 1 → 2 → 3 → 4
2) Run cell 5 to start streaming continuously
3) Press ■ to stop

## 1 — Health Checks

In [1]:
import socket
import time


def wait_for_tcp(host, port, label, timeout=120):
    print(f"Waiting for {label} ({host}:{port})...", end="", flush=True)
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            socket.create_connection((host, port), timeout=2).close()
            print(" OK")
            return
        except OSError:
            print(".", end="", flush=True)
            time.sleep(3)
    raise TimeoutError(f"{label} not ready after {timeout}s")


wait_for_tcp("postgres", 5432, "PostgreSQL")
wait_for_tcp("minio", 9000, "MinIO")
wait_for_tcp("hive-metastore", 9083, "Hive Metastore")
wait_for_tcp("spark-master", 7077, "Spark Master")
wait_for_tcp("kafka", 9092, "Kafka Broker")

print("\nAll required services ready.")

Waiting for PostgreSQL (postgres:5432)... OK
Waiting for MinIO (minio:9000)... OK
Waiting for Hive Metastore (hive-metastore:9083)... OK
Waiting for Spark Master (spark-master:7077)... OK
Waiting for Kafka Broker (kafka:9092)... OK

All required services ready.


## 2 — Spark Session

In [2]:
import os
import time
from pyspark.sql import SparkSession

KAFKA_BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka:9092")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_KEY = os.getenv("MINIO_ROOT_USER", "minio")
MINIO_SECRET = os.getenv("MINIO_ROOT_PASSWORD", "minio123")

DELTA_BASE = "s3a://lakehouse/staging"
CHECKPOINT_BASE = "s3a://lakehouse/checkpoints"
SCHEMA_DIR = os.getenv("SCHEMA_DIR", "/schemas")

DB_HOST = os.getenv("POSTGRES_HOST", "postgres")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB", "thelook")
DB_USER = os.getenv("POSTGRES_USER", "admin")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD", "admin123")

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()
    time.sleep(2)

spark = (
    SparkSession.builder
    .appName("TheLookStreaming")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark session ready.")

Spark session ready.


## 3 — Schemas + Decode Helpers

In [3]:
import pyspark.sql.functions as F
import pyspark.sql.types as T

# ── CDC Envelope schema (Debezium SMT unwraps to this) ──────────────────────
CDC_ENVELOPE_SCHEMA = T.StructType([
    T.StructField("before", T.StringType(), True),
    T.StructField("after",  T.StringType(), True),
    T.StructField("op",     T.StringType(), True),
    T.StructField("ts_ms",  T.StringType(), True),
])

# ── Spark StructTypes — must match exact Kafka wire types from Debezium ──────
# PostgreSQL UUID → JSON string, BIGINT → JSON number
USERS_SCHEMA = T.StructType([
    T.StructField("id",             T.StringType(), True),
    T.StructField("first_name",     T.StringType(), True),
    T.StructField("last_name",      T.StringType(), True),
    T.StructField("email",          T.StringType(), True),
    T.StructField("age",            T.IntegerType(), True),
    T.StructField("gender",        T.StringType(), True),
    T.StructField("street_address",T.StringType(), True),
    T.StructField("postal_code",    T.StringType(), True),
    T.StructField("city",           T.StringType(), True),
    T.StructField("state",          T.StringType(), True),
    T.StructField("country",        T.StringType(), True),
    T.StructField("latitude",       T.DoubleType(), True),
    T.StructField("longitude",      T.DoubleType(), True),
    T.StructField("traffic_source", T.StringType(), True),
    T.StructField("created_at",     T.StringType(), True),
    T.StructField("updated_at",     T.StringType(), True),
])

ORDERS_SCHEMA = T.StructType([
    T.StructField("id",           T.StringType(), True),   # UUID
    T.StructField("user_id",      T.StringType(), True),   # UUID
    T.StructField("status",       T.StringType(), True),
    T.StructField("num_of_items", T.IntegerType(), True),
    T.StructField("created_at",   T.StringType(), True),
    T.StructField("updated_at",   T.StringType(), True),
    T.StructField("returned_at",  T.StringType(), True),
    T.StructField("shipped_at",   T.StringType(), True),
    T.StructField("delivered_at", T.StringType(), True),
    T.StructField("cancelled_at", T.StringType(), True),
])

ORDER_ITEMS_SCHEMA = T.StructType([
    T.StructField("id",           T.StringType(), True),   # UUID
    T.StructField("order_id",     T.StringType(), True),   # UUID
    T.StructField("product_id",   T.LongType(), True),      # BIGINT
    T.StructField("status",       T.StringType(), True),
    T.StructField("quantity",     T.IntegerType(), True),
    T.StructField("sale_price",   T.DoubleType(), True),
    T.StructField("created_at",   T.StringType(), True),
    T.StructField("updated_at",   T.StringType(), True),
    T.StructField("shipped_at",   T.StringType(), True),
    T.StructField("delivered_at", T.StringType(), True),
    T.StructField("returned_at",  T.StringType(), True),
    T.StructField("cancelled_at", T.StringType(), True),
])

EVENTS_SCHEMA = T.StructType([
    T.StructField("id",              T.StringType(), True),   # UUID
    T.StructField("user_id",         T.StringType(), True),   # UUID
    T.StructField("sequence_number", T.IntegerType(), True),
    T.StructField("session_id",      T.StringType(), True),
    T.StructField("ip_address",      T.StringType(), True),
    T.StructField("city",            T.StringType(), True),
    T.StructField("state",           T.StringType(), True),
    T.StructField("postal_code",     T.StringType(), True),
    T.StructField("browser",         T.StringType(), True),
    T.StructField("traffic_source",  T.StringType(), True),
    T.StructField("uri",             T.StringType(), True),
    T.StructField("event_type",     T.StringType(), True),
    T.StructField("created_at",     T.StringType(), True),
])

# Static tables are bootstrapped via JDBC — types come from PostgreSQL (no CDC involved)
REF_PRODUCTS_SCHEMA = T.StructType([
    T.StructField("id",                     T.LongType(), True),      # BIGINT
    T.StructField("cost",                  T.DoubleType(), True),
    T.StructField("category",              T.StringType(), True),
    T.StructField("name",                   T.StringType(), True),
    T.StructField("brand",                  T.StringType(), True),
    T.StructField("retail_price",          T.DoubleType(), True),
    T.StructField("department",            T.StringType(), True),
    T.StructField("sku",                   T.StringType(), True),
    T.StructField("distribution_center_id", T.LongType(), True),   # BIGINT
])

REF_DIST_CENTERS_SCHEMA = T.StructType([
    T.StructField("id",        T.LongType(), True),      # BIGINT
    T.StructField("name",      T.StringType(), True),
    T.StructField("latitude",   T.DoubleType(), True),
    T.StructField("longitude", T.DoubleType(), True),
])

print("StructType schemas ready.")

StructType schemas ready.


## 4 — Bootstrap + Register Static Tables

In [ ]:
import trino

TRINO_HOST = os.getenv("TRINO_HOST", "trino")
TRINO_PORT = "8080"

def trino_exec(sql):
    try:
        conn = trino.dbapi.connect(
            host=TRINO_HOST, port=int(TRINO_PORT),
            user="admin", catalog="delta", schema="staging"
        )
        cur = conn.cursor()
        cur.execute(sql)
        try:
            _ = cur.fetchall()
        except trino.exceptions.TrinoDataError:
            pass
        conn.close()
        return True
    except Exception as e:
        print(f"  WARN: {e}")
        return False

def generate_create_ddl(table_name, schema, location, extra_cols=None):
    """Generate CREATE TABLE IF NOT EXISTS DDL from Spark StructType."""
    col_defs = []
    for f in schema.fields:
        spark_type = f.dataType.simpleString()
        delta_type = _spark_to_delta(spark_type)
        col_defs.append(f"  {f.name} {delta_type}")
    if extra_cols:
        col_defs.extend(extra_cols)
    cols_str = ",\n".join(col_defs)
    return (
        f"CREATE TABLE IF NOT EXISTS {table_name} (\n{cols_str}\n)\n"
        f"WITH (\n  location = '{location}'\n)"
    )

def _spark_to_delta(spark_type):
    m = {
        "string": "VARCHAR",
        "integer": "INTEGER",
        "long": "BIGINT",
        "double": "DOUBLE",
        "boolean": "BOOLEAN",
        "decimal": "DECIMAL",
        "timestamp": "TIMESTAMP",
        "date": "DATE",
    }
    t = spark_type.lower()
    for k, v in m.items():
        if k in t:
            return v
    return "VARCHAR"

def bootstrap(src_table, tgt_path, table_schema, extra_cols=None):
    df = (
        spark.read.format("jdbc")
        .option("url", f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}")
        .option("dbtable", f"public.{src_table}")
        .option("user", DB_USER)
        .option("password", DB_PASSWORD)
        .option("driver", "org.postgresql.Driver")
        .load()
    )
    if "operation" not in df.columns:
        df = df.withColumn("operation", F.lit("r"))
    if "event_ts_ms" not in df.columns:
        df = df.withColumn("event_ts_ms", (F.unix_timestamp(F.current_timestamp()) * 1000).cast("long"))
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(tgt_path)
    count = df.count()
    print(f"  Wrote {count} rows to {tgt_path}")
    return count

# Create staging schema
ok = trino_exec("CREATE SCHEMA IF NOT EXISTS staging")
status = "OK" if ok else "FAIL"
print(f"[{status}] CREATE SCHEMA staging")

# Bootstrap + register static tables
for src_table, table_name, table_schema in [
    ("products",     "ref_products",     REF_PRODUCTS_SCHEMA),
    ("dist_centers", "ref_dist_centers", REF_DIST_CENTERS_SCHEMA),
]:
    tgt_path = f"{DELTA_BASE}/{table_name}"
    bootstrap(src_table, tgt_path, table_schema)
    ddl = generate_create_ddl(
        table_name, table_schema,
        f"{DELTA_BASE}/{table_name}",
        extra_cols=["operation VARCHAR", "event_ts_ms BIGINT"]
    )
    ok = trino_exec(ddl)
    status = "OK" if ok else "FAIL"
    print(f"[{status}] CREATE TABLE staging.{table_name}")

print("Static ref tables ready.")


Static tables (ref_products, ref_dist_centers) are bootstrapped and registered by cell 4.
CDC tables are registered by cell 5 after streams start.

## 5 — Start CDC Streams (continuous)

In [ ]:
# 5a — Raw CDC → Delta (append mode)

# Tune streaming cadence
spark.conf.set("spark.sql.shuffle.partitions", "8")

CDC_TOPICS = {
    "thelook.public.users":       ("users",       f"{DELTA_BASE}/users",       USERS_SCHEMA),
    "thelook.public.orders":       ("orders",       f"{DELTA_BASE}/orders",       ORDERS_SCHEMA),
    "thelook.public.order_items":  ("order_items",  f"{DELTA_BASE}/order_items",  ORDER_ITEMS_SCHEMA),
    "thelook.public.events":       ("events",       f"{DELTA_BASE}/events",       EVENTS_SCHEMA),
}

queries = []
for topic, (table, table_loc, table_schema) in CDC_TOPICS.items():
    raw = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
        .option("subscribe", topic)
        .option("kafka.group.id", "spark-streaming")
        .option("startingOffsets", "earliest")
        .option("failOnDataLoss", "false")
        .option("maxOffsetsPerTrigger", "10000")
        .load()
    )

    envelope_df = raw.selectExpr(
        "CAST(value AS STRING) AS _raw",
        "CAST(key AS STRING) AS kafka_key",
        "CAST(timestamp AS TIMESTAMP) AS kafka_ts",
        "partition", "offset"
    ).select(
        F.from_json(F.col("_raw"), CDC_ENVELOPE_SCHEMA).alias("envelope"),
        F.col("kafka_key"), F.col("kafka_ts"),
        F.col("partition"), F.col("offset"),
    )

    enriched = envelope_df.select(
        F.from_json(F.col("envelope.after"), table_schema).alias("data"),
        F.col("envelope.op").alias("op"),
        F.col("envelope.ts_ms").alias("ts_ms"),
        F.col("kafka_key"), F.col("kafka_ts"),
        F.col("partition"), F.col("offset"),
    )

    filtered = enriched.filter(F.col("data.id").isNotNull())

    data_cols = [F.col(f"data.{f.name}").alias(f.name) for f in table_schema.fields]
    final = filtered.select(*data_cols, "kafka_ts", "partition", "offset", "op", "ts_ms")

    q = (
        final.writeStream.format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/{table}")
        .option("mergeSchema", "true")
        .trigger(processingTime="30 seconds")
        .start(f"{DELTA_BASE}/{table}")
    )
    queries.append(q)
    print(f"Stream started: {topic} -> {DELTA_BASE}/{table}")

print("\nRegistering CDC tables via Trino...")
for table, table_loc, table_schema in CDC_TOPICS.values():
    extra_cols = ["kafka_ts TIMESTAMP", "partition INTEGER", "offset BIGINT", "op VARCHAR", "ts_ms VARCHAR"]
    ddl = generate_create_ddl(table, table_schema, table_loc, extra_cols=extra_cols)
    ok = trino_exec(ddl)
    status = "OK" if ok else "WARN"
    print(f"  [{status}] CREATE TABLE staging.{table}")

print("\nHMS registration done.")
print(f"\nAll {len(queries)} streams running (micro-batch every 30s). Press stop.")
spark.streams.awaitAnyTermination()

### 5b — foreachBatch deduplication → Hive tables

Demonstrates per-batch deduplication using `ROW_NUMBER` over the Kafka topic, then writing to Hive tables (partitioned by year). This pattern lets you deduplicate **within each micro-batch** rather than appending everything and relying on downstream dbt deduplication.

> For the lakehouse pipeline, append-to-Delta (cell 5a) + dbt deduplication is preferred. This cell shows `foreachBatch` for cases where Hive table access or per-batch aggregation is needed.

In [ ]:
# 5b — foreachBatch deduplication → Hive tables

# Ensure staging schema exists in Hive Metastore
spark.sql("CREATE DATABASE IF NOT EXISTS staging_hive")

CDC_TOPICS_BATCH = {
    "thelook.public.users":       ("users_hive",       f"{DELTA_BASE}/hive/users",       USERS_SCHEMA,       "created_at"),
    "thelook.public.orders":       ("orders_hive",       f"{DELTA_BASE}/hive/orders",       ORDERS_SCHEMA,       "created_at"),
    "thelook.public.order_items":  ("order_items_hive",  f"{DELTA_BASE}/hive/order_items",  ORDER_ITEMS_SCHEMA,  "created_at"),
    "thelook.public.events":       ("events_hive",       f"{DELTA_BASE}/hive/events",       EVENTS_SCHEMA,       "created_at"),
}

batch_queries = []


def make_foreach_batch_fn(table_name, table_loc, partition_col, year_partition_col):
    """Build a foreachBatch function that deduplicates per batch then upserts to Hive."""
    def write_batch(batch_df, batch_id):
        # Deduplicate within this micro-batch: keep latest record per entity key
        deduped = (
            batch_df
            .withColumn("rn", F.row_number().over(
                F.Window.partitionBy("id").orderBy(F.col("kafka_ts").desc())
            ))
            .filter(F.col("rn") == 1)
            .drop("rn")
        )

        # Partition column (must exist in deduped DataFrame)
        if partition_col in deduped.columns:
            # Extract year from partition column
            deduped = deduped.withColumn(
                year_partition_col,
                F.year(F.to_timestamp(F.col(partition_col)))
            )

        deduped.write.format("delta").mode("append").partitionBy(year_partition_col).save(table_loc)
        print(f"  Batch {batch_id}: wrote {deduped.count()} deduped rows to {table_loc}")
    return write_batch


for topic, (table_name, table_loc, table_schema, partition_col) in CDC_TOPICS_BATCH.items():
    raw = (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
        .option("subscribe", topic)
        .option("kafka.group.id", "spark-streaming-batch")
        .option("startingOffsets", "earliest")
        .option("failOnDataLoss", "false")
        .option("maxOffsetsPerTrigger", "10000")
        .load()
    )

    envelope_df = raw.selectExpr(
        "CAST(value AS STRING) AS _raw",
        "CAST(key AS STRING) AS kafka_key",
        "CAST(timestamp AS TIMESTAMP) AS kafka_ts",
        "partition", "offset"
    ).select(
        F.from_json(F.col("_raw"), CDC_ENVELOPE_SCHEMA).alias("envelope"),
        F.col("kafka_key"), F.col("kafka_ts"),
        F.col("partition"), F.col("offset"),
    )

    enriched = envelope_df.select(
        F.from_json(F.col("envelope.after"), table_schema).alias("data"),
        F.col("envelope.op").alias("op"),
        F.col("envelope.ts_ms").alias("ts_ms"),
        F.col("kafka_key"), F.col("kafka_ts"),
        F.col("partition"), F.col("offset"),
    )

    filtered = enriched.filter(F.col("data.id").isNotNull())
    data_cols = [F.col(f"data.{f.name}").alias(f.name) for f in table_schema.fields]
    final = filtered.select(*data_cols, "kafka_ts", "partition", "offset", "op", "ts_ms")

    year_col = table_name.replace("_hive", "_year")
    batch_fn = make_foreach_batch_fn(table_name, table_loc, partition_col, year_col)

    q = (
        final.writeStream
        .foreachBatch(batch_fn)
        .option("checkpointLocation", f"{CHECKPOINT_BASE}/{table_name}")
        .trigger(processingTime="30 seconds")
        .outputMode("update")
        .start()
    )
    batch_queries.append(q)
    print(f"foreachBatch stream started: {topic} -> {table_name}")

print(f"\nAll {len(batch_queries)} foreachBatch streams running.")
print("Press stop to terminate.")
spark.streams.awaitAnyTermination()